In [4]:
# Convert into pandas data frame
import pandas as pd
from io import StringIO

data = """order_id,customer_segment,region,days_since_signup,items_count,avg_item_price,used_coupon,total_value
1,Consumer,North,45,2,30.0,Yes,54.0
2,Corporate,South,400,5,18.0,No,90.0
3,Home Office,East,120,1,200.0,Yes,170.0
4,Consumer,West,10,3,25.0,No,75.0
5,Corporate,North,250,4,40.0,Yes,150.0
6,Consumer,South,60,1,300.0,No,300.0
7,Home Office,West,15,2,80.0,Yes,136.0
8,Corporate,East,500,6,15.0,No,90.0
9,Consumer,North,5,1,100.0,Yes,85.0
10,Home Office,South,220,3,50.0,No,150.0"""

df = pd.read_csv(StringIO(data))
df.head(10)


,order_id,customer_segment,region,days_since_signup,items_count,avg_item_price,used_coupon,total_value
0,1,Consumer,North,45,2,30.0,Yes,54.0
1,2,Corporate,South,400,5,18.0,No,90.0
2,3,Home Office,East,120,1,200.0,Yes,170.0
3,4,Consumer,West,10,3,25.0,No,75.0
4,5,Corporate,North,250,4,40.0,Yes,150.0
5,6,Consumer,South,60,1,300.0,No,300.0
6,7,Home Office,West,15,2,80.0,Yes,136.0
7,8,Corporate,East,500,6,15.0,No,90.0
8,9,Consumer,North,5,1,100.0,Yes,85.0
9,10,Home Office,South,220,3,50.0,No,150.0


Tasks:

Load and inspect data:

Read the CSV using pandas.

Print basic info and a quick description (e.g., df.info(), df.describe()).

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           10 non-null     int64  
 1   customer_segment   10 non-null     object 
 2   region             10 non-null     object 
 3   days_since_signup  10 non-null     int64  
 4   items_count        10 non-null     int64  
 5   avg_item_price     10 non-null     float64
 6   used_coupon        10 non-null     int64  
 7   total_value        10 non-null     float64
dtypes: float64(2), int64(4), object(2)
memory usage: 772.0+ bytes


In [4]:
df.describe()

,order_id,days_since_signup,items_count,avg_item_price,total_value
count,10.00000,10.000000,10.00000,10.00000,10.000000
mean,5.50000,162.500000,2.80000,85.80000,130.000000
std,3.02765,175.503245,1.75119,93.66465,70.994522
min,1.00000,5.000000,1.00000,15.00000,54.000000
25%,3.25000,22.500000,1.25000,26.25000,86.250000
50%,5.50000,90.000000,2.50000,45.00000,113.000000
75%,7.75000,242.500000,3.75000,95.00000,150.000000
max,10.00000,500.000000,6.00000,300.00000,300.000000


Feature engineering:

Convert used_coupon to a numeric feature (e.g., 0/1).

Optionally create one additional feature you think might help (for example, estimated “planned_value” = items_count * avg_item_price if that’s not already equal to total_value).

In [5]:
df['used_coupon'] = df['used_coupon'].map({'Yes':1,'No':0})
df.head(10)

,order_id,customer_segment,region,days_since_signup,items_count,avg_item_price,used_coupon,total_value
0,1,Consumer,North,45,2,30.0,1,54.0
1,2,Corporate,South,400,5,18.0,0,90.0
2,3,Home Office,East,120,1,200.0,1,170.0
3,4,Consumer,West,10,3,25.0,0,75.0
4,5,Corporate,North,250,4,40.0,1,150.0
5,6,Consumer,South,60,1,300.0,0,300.0
6,7,Home Office,West,15,2,80.0,1,136.0
7,8,Corporate,East,500,6,15.0,0,90.0
8,9,Consumer,North,5,1,100.0,1,85.0
9,10,Home Office,South,220,3,50.0,0,150.0


In [6]:
'''Optionally create one additional feature you think might help (for example, estimated “planned_value” = items_count * avg_item_price if that’s not already equal to total_value).'''

df['planned_value'] = df['items_count'] * df['avg_item_price']
df.head()

,order_id,customer_segment,region,days_since_signup,items_count,avg_item_price,used_coupon,total_value,planned_value
0,1,Consumer,North,45,2,30.0,1,54.0,60.0
1,2,Corporate,South,400,5,18.0,0,90.0,90.0
2,3,Home Office,East,120,1,200.0,1,170.0,200.0
3,4,Consumer,West,10,3,25.0,0,75.0,75.0
4,5,Corporate,North,250,4,40.0,1,150.0,160.0


In [7]:
'''Use scikit‑learn train_test_split (e.g., 70/30 split).

Choose at least items_count, avg_item_price, days_since_signup, customer_segment, region, and used_coupon (plus your new feature if you make one).'''
selected_features = ['items_count','avg_item_price','days_since_signup','customer_segment','region','used_coupon','planned_value']

from sklearn.model_selection import train_test_split
X = df[selected_features]
y= df['total_value']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state = 42)



Model & pipeline:
Use a simple model, e.g., LinearRegression or RandomForestRegressor.
Build a scikit‑learn Pipeline that includes:
Appropriate preprocessing for categorical vs numeric features (e.g., ColumnTransformer with One‑Hot Encoding for categoricals).
The regression model.

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = ['customer_segment','region']
numerical_features = ['items_count','avg_item_price']

#Transformation
categorical_transformation = OneHotEncoder(handle_unknown = 'ignore')
numerical_transformation = StandardScaler()

#Combining the preprocessing techniques using ColumnTransformer
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    transformers = [('Numbers',numerical_transformation, numerical_features),
                    ('Categorical', categorical_transformation,categorical_features)])
steps = [('preprocessing',preprocessor),
 ('Regression',RandomForestRegressor(n_estimators=100,random_state=42)) ]
pipe1 = Pipeline(steps)
pipe1.fit(X_train,y_train)
y_pred = pipe1.predict(X_test)


In [15]:
print(y_pred)

[128.53  98.65 134.8 ]


Evaluate:
Calculate RMSE (root mean squared error) on the test set.
Print predicted vs actual total_value for the test rows.

In [16]:
import numpy as np
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test,y_pred)

rmse = np.sqrt(mse)
print(rmse)

98.76018664758925


In [17]:
print('The values are {} for Predicted'.format(y_pred))
print('The values are {} for Actual'.format(y_test))

The values are [128.53  98.65 134.8 ] for Predicted
The values are 8     85.0
1     90.0
5    300.0
Name: total_value, dtype: float64 for Actual
